In [ ]:
# Cell 1 — Imports & setup
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from src.macro_regime.build import build_master_df
from src.macro_regime.regimes import (
    assign_inflation_regime,
    assign_rate_regime,
    combine_macro_regime,
    classify_vix_stress_regime,
)

In [ ]:
# Cell 2 — Load data, assign regimes, compute rolling volatility
from src.macro_regime.stats import rolling_volatility

df = build_master_df("../data/processed")
df = assign_inflation_regime(df)
df = assign_rate_regime(df)
df = combine_macro_regime(df)
df["vix_regime"] = classify_vix_stress_regime(df["vix"])

# Compute 20-day rolling volatility per sector
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
df["rolling_vol"] = (
    df.groupby("ticker")["sector_return"]
    .transform(lambda s: rolling_volatility(s, window=20))
)

SECTOR_NAMES = {
    "XLB": "Materials", "XLE": "Energy", "XLF": "Financials",
    "XLI": "Industrials", "XLK": "Technology", "XLP": "Cons. Staples",
    "XLU": "Utilities", "XLV": "Health Care", "XLY": "Cons. Discr.",
    "XLRE": "Real Estate", "XLC": "Comm. Services",
}
df["sector_name"] = df["ticker"].map(SECTOR_NAMES)

REGIME_ORDER = [
    "High Inflation / Rising Rates",
    "High Inflation / Falling Rates",
    "Low Inflation / Rising Rates",
    "Low Inflation / Falling Rates",
]

plot_df = df.dropna(subset=["rolling_vol", "macro_regime"]).copy()
print(f"Rows with valid vol + regime: {len(plot_df):,}")
print(f"Regime counts:\n{plot_df['macro_regime'].value_counts()}")

In [ ]:
# Cell 3 — Grouped bar chart: mean volatility by regime x sector
mean_vol = (
    plot_df.groupby(["macro_regime", "sector_name"])["rolling_vol"]
    .mean()
    .unstack("sector_name")
    .reindex(index=REGIME_ORDER)
)

fig, ax = plt.subplots(figsize=(14, 6))
mean_vol.plot.bar(ax=ax, width=0.8, edgecolor="white", linewidth=0.5)

ax.set_title(
    "Mean 20-Day Rolling Volatility by Macro Regime",
    fontsize=15, fontweight="bold", pad=15,
)
ax.set_ylabel("Rolling Volatility (daily std dev)", fontsize=12)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=25, labelsize=10)
ax.legend(
    title="Sector", bbox_to_anchor=(1.02, 1), loc="upper left",
    fontsize=9, title_fontsize=10,
)

plt.tight_layout()
plt.savefig("../docs/vol_bar_by_regime.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 4 — Box plots: volatility distributions by regime (faceted by sector)
g = sns.catplot(
    data=plot_df,
    x="macro_regime",
    y="rolling_vol",
    col="sector_name",
    col_wrap=4,
    kind="box",
    order=REGIME_ORDER,
    height=3.2,
    aspect=1.1,
    showfliers=False,
    palette="Set2",
)

g.set_titles("{col_name}", fontsize=11, fontweight="bold")
g.set_axis_labels("", "Rolling Volatility")

for ax in g.axes.flat:
    ax.set_xticklabels(
        ["HI / RR", "HI / FR", "LI / RR", "LI / FR"],
        fontsize=8,
    )

g.figure.suptitle(
    "Sector Volatility Distributions Across Macro Regimes",
    fontsize=15, fontweight="bold", y=1.02,
)

plt.tight_layout()
plt.savefig("../docs/vol_boxplots_by_sector.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 5 — Overlaid violin plot: all sectors on one axis, split by regime
fig, ax = plt.subplots(figsize=(16, 6))

sns.violinplot(
    data=plot_df,
    x="sector_name",
    y="rolling_vol",
    hue="macro_regime",
    hue_order=REGIME_ORDER,
    order=[SECTOR_NAMES[t] for t in SECTOR_NAMES],
    cut=0,
    inner="quart",
    linewidth=0.6,
    palette="coolwarm",
    ax=ax,
)

ax.set_title(
    "Sector Volatility Distributions by Macro Regime (Violin)",
    fontsize=15, fontweight="bold", pad=15,
)
ax.set_ylabel("20-Day Rolling Volatility", fontsize=12)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.legend(
    title="Regime", bbox_to_anchor=(1.02, 1), loc="upper left",
    fontsize=9, title_fontsize=10,
)

plt.tight_layout()
plt.savefig("../docs/vol_violin_by_regime.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 6 — Summary table: mean & median vol by regime x sector
summary = (
    plot_df.groupby(["macro_regime", "sector_name"])["rolling_vol"]
    .agg(["mean", "median", "std", "count"])
    .round(6)
    .unstack("sector_name")
    .reindex(index=REGIME_ORDER)
)

summary